In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:
# Q6: Churn rate by Gender, Senior Citizen, Partner, Dependents
churn_dimensions = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

results = []
for dim in churn_dimensions:              # loop over each demographic column one at a time
    group = df.groupby(dim)['Churn'].apply(
        # for each group value (e.g. Male / Female): count rows where Churn == 'Yes',
        # divide by total rows in that group, multiply by 100 → churn rate %
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).reset_index()
    group.columns = ['value', 'churn_rate']  # rename: first col = the group value, second = its churn rate
    group['dimension'] = dim                 # tag this mini-table so we know which demographic it came from
    results.append(group)

# stack all four mini-tables (one per demographic) into one unified table
report = pd.concat(results).sort_values('churn_rate', ascending=False)
report['churn_rate'] = report['churn_rate'].round(2)
report[['dimension', 'value', 'churn_rate']]

In [ ]:
# Q7: Customer segments based on tenure and churn analysis
df['tenure_segment'] = pd.cut(
    df['tenure'],                     # continuous column: months as a customer (1 to 72)
    bins=[0, 12, 24, 48, 72],         # cut points — creates 4 ranges: 0-12, 12-24, 24-48, 48-72
    # like sorting exam scores into grade buckets — each customer gets a readable label
    labels=['New (0-12)', 'Growing (12-24)', 'Mature (24-48)', 'Loyal (48-72)']
)

# groupby().agg() computes multiple stats per segment in one pass
# each key = output column name, each value = (source_column, aggregation_function)
segment_analysis = df.groupby('tenure_segment').agg(
    customer_count = ('customerID', 'count'),           # total customers in this bucket
    churned        = ('Churn', lambda x: (x == 'Yes').sum()),  # how many of them churned
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
    # churn_rate = churned count ÷ total in bucket × 100
).reset_index()

segment_analysis

In [ ]:
# Q8: Top 10 customer profiles most likely to churn
profiles = df.groupby(['Contract', 'PaymentMethod', 'InternetService']).agg(
    customer_count = ('customerID', 'count'),
    churned = ('Churn', lambda x: (x == 'Yes').sum()),
    churn_rate = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index()

profiles = profiles[profiles['customer_count'] >= 30]

profiles.sort_values('churn_rate', ascending=False).head(10)

,Contract,PaymentMethod,InternetService,customer_count,churned,churn_rate
7,Month-to-month,Electronic check,Fiber optic,1307,789,60.37
10,Month-to-month,Mailed check,Fiber optic,201,102,50.75
1,Month-to-month,Bank transfer (automatic),Fiber optic,327,149,45.57
4,Month-to-month,Credit card (automatic),Fiber optic,293,122,41.64
6,Month-to-month,Electronic check,DSL,474,192,40.51
9,Month-to-month,Mailed check,DSL,367,113,30.79
3,Month-to-month,Credit card (automatic),DSL,185,50,27.03
19,One year,Electronic check,Fiber optic,196,51,26.02
11,Month-to-month,Mailed check,No,325,67,20.62
2,Month-to-month,Bank transfer (automatic),No,65,13,20.00


In [ ]:
# Q9: Calculate CLV and compare churned vs retained customers
df['CLV'] = df['MonthlyCharges'] * df['tenure']

clv_comparison = df.groupby('Churn').agg(
    customer_count = ('customerID', 'count'),
    avg_clv = ('CLV', lambda x: round(x.mean(), 2)),
    avg_tenure = ('tenure', lambda x: round(x.mean(), 2)),
    avg_monthly = ('MonthlyCharges', lambda x: round(x.mean(), 2))
).reset_index()

clv_comparison

,Churn,customer_count,avg_clv,avg_tenure,avg_monthly
0,No,5174,2549.77,37.57,61.27
1,Yes,1869,1531.61,17.98,74.44


In [7]:
# Q10: Customer Risk Score using Contract, Tenure, Tech Support, Online Security
def calculate_risk_score(row):
    score = 0

    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1
    else:
        score += 0

    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1
    else:
        score += 0

    if row['TechSupport'] == 'No':
        score += 2

    if row['OnlineSecurity'] == 'No':
        score += 2

    return score

df['risk_score'] = df.apply(calculate_risk_score, axis=1)

df.groupby('risk_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})

,risk_score,churn_rate
0,0,2.39
1,1,4.33
2,2,4.74
3,3,8.76
4,4,10.40
5,5,15.42
6,6,24.85
7,7,36.41
8,8,41.02
9,9,45.96
